# 第9周 · 作业：正式研究实验 1（预计约7小时）

**研究问题**：增加网络深度是否提升 FashionMNIST 分类性能？在小样本下，结论是否改变？

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 实验设计：超参数分类 + 预先写下的假设 | 1 小时 |
| B | 执行：4 种深度 × 2 种数据量 × 2 个学习率 × 3 个种子 + 随机森林基线 | 2.5 小时 |
| C | 结果表与误差棒图 + 分析问题 | 1.5 小时 |
| D | 实验记录（正式提交格式） | 1 小时 |
| E | 代码讲解 | 1 小时 |

**规则**：**从本周起可以使用 AI 编程助手写代码**，但必须完成 E 部分的"代码讲解"段落（用自己的话解释关键代码，并注明哪些部分由 AI 生成），且可能被随机抽查 5 分钟口头讲解。

**评分**（研究实验评分表）：设计 30% · 执行 30% · 分析与解释 30% · 表达 10%。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from zero2researcher import data, experiments
from zero2researcher.checks import week09 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 实验设计（先写，再跑）

实验规格：

- **深度** ∈ {1, 2, 4, 8}：隐层个数，每层 128 个神经元、ReLU，输出层 10 维 logits；
- **训练集规模** ∈ {2000, 10000}：FashionMNIST 训练集的**前** 2000 / 10000 张（固定索引）；
- **学习率** ∈ {1e-3, 3e-3}：Adam，**对每个 (深度, 规模) 组合单独选**——选法：3 个种子的**验证准确率均值**最高者；
- **固定**：2 epochs、batch 64、默认初始化；验证集 = 训练集第 50000–51999 张（2000 张，不与任何训练子集重叠）；测试集 = 全部 10000 张；
- **种子** 0、1、2；报告选定学习率下 3 个种子的**测试**准确率均值 ± 标准差；
- **基线**：同样两种数据量上的 `RandomForestClassifier(n_estimators=100, random_state=seed)`，3 个种子。

在运行任何代码之前，填写下面两项（**预注册**：先写下预期，跑完之后不许改——跑完再改的"假设"不叫假设）：

**A1 · 超参数分类**（把本实验涉及的超参数填进三类，每类至少写两个；冗余超参数写明"如何调"）

- 科学超参数：
- 冗余超参数：
- 固定超参数：

**A2 · 假设**（你预期深度怎样影响测试准确率？两种数据量下是否不同？什么样的结果会推翻你的假设？）

- 

## B · 执行实验

数据准备已给出（张量一次算好，训练时不再逐张变换，CPU 上也很快）：

In [ ]:
def load_dataset(cls, train):
    # 下载并缓存（FashionMNIST 约 30MB，只有第一次要等）；网络抖动时自动重试一次
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


fm_train = load_dataset(datasets.FashionMNIST, train=True)
fm_test = load_dataset(datasets.FashionMNIST, train=False)
pixels = fm_train.data.float() / 255.0
MEAN, STD = pixels.mean().item(), pixels.std().item()      # 第 8 周的"魔法数字"算法


def to_inputs(images):
    # uint8 图像 (N, 28, 28) → 标准化并展平的 float 张量 (N, 784)：一次算好，训练时不再逐张变换
    return ((images.float() / 255.0 - MEAN) / STD).reshape(len(images), -1)


X_val, y_val = to_inputs(fm_train.data[50000:52000]), fm_train.targets[50000:52000]   # 固定验证集
X_test, y_test = to_inputs(fm_test.data), fm_test.targets                               # 测试集全量
print(f"mean = {MEAN:.4f}, std = {STD:.4f} | 验证 {len(X_val)} 张 | 测试 {len(X_test)} 张")

X_pool, y_pool = to_inputs(fm_train.data[:10000]), fm_train.targets[:10000]   # 前 2000 张是前 10000 张的子集
criterion = nn.CrossEntropyLoss()


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X.to(device)).argmax(dim=1).cpu() == y).float().mean().item()

**B1** · 写 `make_mlp(depth, width=128)`：`depth` 个 `Linear + ReLU` 的隐层，再接一个输出 10 维的 `Linear`。例如 `make_mlp(2)` 应是 784-128-128-10。

In [ ]:
def make_mlp(depth, width=128):
    # 在这里写你的代码
    raise NotImplementedError


print(make_mlp(2))
print("depth 8 的参数量:", sum(p.numel() for p in make_mlp(8).parameters()))

**B2** · 写 `run_config(depth, size, lr, seed, epochs=2)`：固定种子 → 建模型 → 在前 `size` 张训练图上用 Adam(lr) 训练 `epochs` 个 epoch（batch 64，五步核心）→ 返回 `(验证准确率, 测试准确率)`。

In [ ]:
def run_config(depth, size, lr, seed, epochs=2):
    # 在这里写你的代码
    raise NotImplementedError


print("冒烟测试 (depth=1, size=2000, lr=1e-3, seed=0):", run_config(1, 2000, 1e-3, 0))

**B3** · 主循环：对每个 (规模, 深度)，在两个学习率上各跑 3 个种子；按**验证准确率均值**选出最好的学习率；把该学习率下 3 个种子的**测试**准确率存进 `results`。要求的结构（自查会检查）：

```python
results[(size, depth)] = {"best_lr": ..., "accs": [3 个测试准确率], "mean": ..., "std": ...}
```

`mean, std` 用 `experiments.mean_std(accs)` 算。顺便把每个学习率的验证准确率均值存进 `results[(size, depth)]["val_by_lr"]`，写分析时要用。

In [ ]:
SIZES, DEPTHS, LRS, SEEDS = (2000, 10000), (1, 2, 4, 8), (1e-3, 3e-3), (0, 1, 2)
results = {}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"全部 {len(SIZES) * len(DEPTHS) * len(LRS) * len(SEEDS)} 次训练用时 {time.time() - t0:.1f} 秒")
for (size, depth), r in results.items():
    print(f"size {size:5d} | depth {depth} | best lr {r['best_lr']:g} | test {experiments.format_result(r['accs'])}")
ck.depth_experiment(results)

**B4** · 随机森林基线：两种数据量 × 3 个种子，输入为展平像素（标准化与否对树模型无影响，直接用 `X_pool`），测试集相同。结果存进 `rf_results[size] = [3 个测试准确率]`。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_results = {}
# 在这里写你的代码
raise NotImplementedError
for size, accs in rf_results.items():
    print(f"RandomForest, size {size}: {experiments.format_result(accs)}")

## C · 结果表与误差棒图

**C1** · 做一张结果表 `table`（pandas DataFrame）：行 = 深度（外加一行 "RF"），列 = 两种数据量，格子里写 `均值 ± 标准差`，MLP 的格子再注明选中的学习率。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**C2** · 画一张带误差棒的图：横轴深度（1, 2, 4, 8），纵轴测试准确率，两种数据量各一条线（误差棒 = 标准差），随机森林画成同色的水平虚线。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**C3 · 分析问题**（每题两三句话，引用你表里的数字）

1. 在每种数据量下，深度对测试准确率的效应是什么方向？差异是否大于标准差（用第 6 周的三分法下结论）？
2. 小样本（2000 张）与大样本（10000 张）下结论是否不同？随机森林基线在哪种规模下更难被超越？
3. 看 `val_by_lr`：不同深度选中的学习率一样吗？如果所有深度都只用 1e-3，结论会变吗？这说明了 Playbook 的哪条原则？
4. 8 层网络表现最差——除了"深度本身有害"，至少再给出一个**由固定超参数造成**的解释（提示：2 epochs、默认初始化、没有 BatchNorm；回想 S1 的激活直方图）。你会设计什么后续实验区分这些解释？

（在这里回答）

## D · 实验记录（正式提交）

按第 6 周起的实验记录格式填写。"结果"一栏必须是均值 ± 标准差并引用 C 部分的表和图；"局限"一栏必须讨论固定超参数对结论的限定。

## 实验记录 #2 · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## E · 代码讲解（本周起每份作业必交）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 `run_config` 和主循环在做什么——尤其是**学习率是在哪个数据集上、按什么指标选的，为什么不能用测试集选**；
2. 注明哪些代码由 AI 助手生成、你对它做了哪些检查或修改（例如：五步顺序、`model.eval()`、损失函数的输入、种子是否在建模型之前设置）；
3. 写出一处你一开始没看懂、后来弄懂的地方。

讲不清楚的代码视同不是你写的——口头抽查就从这一段问起。

（在这里写）